# M10 Part A.5 — run MAMMA on BEHAVE → SMPL-X exports (Drive-persisted)

Produces per-frame SMPL-X exports on Drive; `M10 - surfel_avatar v2` / `M10 - smpl_person_render` consume them with `--smpl_source mamma`.

**Everything lives on Drive** (footage, weights, body-models, MAMMA outputs, exports) and the built conda env is snapshotted to Drive — so a runtime timeout doesn't cost the ~20-min compile, the gated downloads, or any pipeline work. Slower Drive I/O, but durable.

> **On reconnect:** just Run-all. §3 restores the env from Drive (fast) instead of rebuilding; §4 skips already-downloaded weights; §6 outputs land on Drive.
>
> **EXPORT-ONLY quick path** (MAMMA already ran; you just want fresh per-frame exports, e.g. after an exporter change): run **§1 → §2 → §7** and skip §3–§6 entirely — the exporter is numpy-only and reads the ma_3d outputs already on Drive.
>
> **Conventions (SOLVED 2026-07-16, do not re-guess):** calib extrinsics = **cam2world**, quaternion = **wxyz** (`world2cam` was the first wrong guess → scrambled world poses). Exports now carry `vertices` + **hand/jaw/eye poses** (dropping hands caused flat-mannequin-hand misregistration in the avatar).

## 1. GPU + mount + clone

In [ ]:
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')
import os, sys
os.chdir('/content')
for repo, url in [('gaussian-surfel-local-map','https://github.com/steptang/gaussian-surfel-local-map.git'),
                  ('behave-dataset','https://github.com/xiexh20/behave-dataset.git'),
                  ('mamma','https://github.com/cuevhv/mamma.git')]:
    if not os.path.isdir(f'/content/{repo}'):
        !git clone -q {url}
!cd /content/gaussian-surfel-local-map && git pull -q
sys.path.insert(0, '/content/gaussian-surfel-local-map')
sys.path.insert(0, '/content/behave-dataset')
!pip -q install pyyaml opencv-python-headless
print('repos ready')

## 2. Paths — everything durable under `MRUN` on Drive

In [ ]:
DRIVE    = '/content/drive/MyDrive/Research'
SEQ      = 'Date03_Sub05_boxtiny'
SEQ_DIR  = f'{DRIVE}/datasets/behave/sequences/{SEQ}'   # raw BEHAVE (t*.000)
N_SELECT = 24                                           # MUST match the converter (frame i == timestep_i)
MRUN     = f'{DRIVE}/2dgs_output/mamma_run'             # <-- all durable artifacts live here
MAMMA_IN = f'{MRUN}/input'                              # footage (videos_crf24) + calib + capture
WEIGHTS  = f'{MRUN}/weights'                            # -> mamma/data/weights (persist gated dl)
BODYM    = f'{MRUN}/body_models'                        # -> mamma/data/body_models (SMPL-X)
MOUT     = f'{MRUN}/output'                             # MAMMA out_dir (ma_cap.../ma_3d/ma_vis)
# Export lineage: `{SEQ}` = BAD (world2cam calib); `{SEQ}_cam2world` = good, no hands;
# `{SEQ}_c2w_hands` = BAD (split from the world2cam ma_3d output by mistake — 45cm vertex diff);
# `{SEQ}_c2w_hands2` = current (cam2world ma_3d + vertices + hand/jaw poses).
EXPORTS  = f'{DRIVE}/datasets/mamma_exports/{SEQ}_c2w_hands2'
ENV_TAR  = f'{MRUN}/mamma_env.tar.gz'                   # snapshot of the built conda env
FORK     = '/content/gaussian-surfel-local-map'
MM       = '/content/mamma/bin/micromamba'; ROOT = '/content/mm'
import os
for d in (MRUN, MAMMA_IN, WEIGHTS, BODYM, MOUT, EXPORTS): os.makedirs(d, exist_ok=True)
assert os.path.isdir(SEQ_DIR), SEQ_DIR; print('paths ready under', MRUN, '| exports ->', EXPORTS)

## 3. Env — restore from Drive if present, else build (~20 min) and snapshot to Drive
Working recipe: `cuda-toolkit` 12.4.1 in the env + `FORCE_CUDA`/`TORCH_CUDA_ARCH_LIST=8.0` + capped `MAX_JOBS` (else the detectron2/pytorch_sdf CUDA build OOM-kills with an empty error), then re-pin `iopath==0.1.10`.

In [ ]:
%cd /content/mamma
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
import os; os.makedirs(ROOT, exist_ok=True)
if os.path.exists(ENV_TAR):
    print('restoring env from Drive (skips the ~20-min compile)…')
    !tar -xzf "$ENV_TAR" -C /content/mm
else:
    print('building env (~20 min)…')
    !$MM create -y -r $ROOT -f requirements/mamma_conda.yaml
    !$MM install -y -r $ROOT -n mamma -c nvidia/label/cuda-12.4.1 -c conda-forge cuda-toolkit
    !$MM run -r $ROOT -n mamma bash -lc 'export CUDA_HOME=$CONDA_PREFIX PATH=$CONDA_PREFIX/bin:$PATH \
        FORCE_CUDA=1 TORCH_CUDA_ARCH_LIST="8.0" MAX_JOBS=2 NVCC_APPEND_FLAGS="--threads 2"; \
      pip install -r requirements/requirements.txt && \
      pip install --no-build-isolation -r requirements/requirements_no_build_isolation.txt && \
      pip install "iopath==0.1.10"'
    print('snapshotting env to Drive (one-time, ~few min)…')
    !tar --exclude=./pkgs -czf "$ENV_TAR" -C /content/mm .
print('env ready')

In [ ]:
# 3b. conda->micromamba shim (MAMMA hardcodes `conda run -n mamma`) + Drive-link weights/body_models
shim = '/content/mm/envs/mamma/bin/conda'
open(shim,'w').write('''#!/usr/bin/env bash
MM=/content/mamma/bin/micromamba; ROOT=/content/mm
if [ "$1" = "run" ]; then
  shift; env=""; argv=()
  while [ $# -gt 0 ]; do case "$1" in
    -n|--name) env="$2"; shift 2;;
    --no-capture-output|--live-stream) shift;;
    *) argv+=("$1"); shift;; esac; done
  exec "$MM" run -r "$ROOT" -n "$env" "${argv[@]}"
fi
exec "$MM" "$@"
''')
import os; os.chmod(shim, 0o755)
# IMPORTANT: copy repo-bundled data assets (e.g. downsampled_verts/smplx_512_body_parts.npz, which
# ma_3d needs) into Drive BEFORE symlinking, else the symlink hides them. -n = don't clobber downloads.
!cp -rn /content/mamma/data/weights/. "$WEIGHTS/" 2>/dev/null || true
!cp -rn /content/mamma/data/body_models/. "$BODYM/" 2>/dev/null || true
!rm -rf /content/mamma/data/weights /content/mamma/data/body_models
!ln -s "$WEIGHTS" /content/mamma/data/weights
!ln -s "$BODYM"  /content/mamma/data/body_models
print('shim + Drive-linked weights/body_models ready (repo assets copied in)')

## 4. Weights (staged on Drive; skips if already there) — gated ones need YOUR license sign-in

In [ ]:
%cd /content/mamma
!mkdir -p data/weights/yolo data/weights/sam2
![ -f data/weights/yolo/yolo12x.pt ] || wget -q -O data/weights/yolo/yolo12x.pt https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo12x.pt
![ -f data/weights/sam2/sam2.1_hiera_large.pt ] || wget -q -O data/weights/sam2/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
# GATED (accept licenses first; downloads persist on Drive so this is one-time):
# !$MM run -r $ROOT -n mamma bash data/download_mamma_weights.sh --all
# !$MM run -r $ROOT -n mamma bash data/download_smplx_locked_head.sh
print('weights staged on Drive; run the two gated scripts once (uncomment) after accepting licenses')

In [ ]:
!$MM run -r $ROOT -n mamma python -m inference doctor

## 5. BEHAVE → MAMMA footage (per-camera mp4s + calib + capture) — written to Drive

In [ ]:
!rm -rf "$MAMMA_IN/$SEQ"
!cd $FORK && PYTHONPATH=$FORK:/content/behave-dataset python -m tracking.behave_to_mamma input \
  --seq_dir "$SEQ_DIR" --out_root "$MAMMA_IN" --seq_name "$SEQ" --n_select $N_SELECT \
  --extrinsics cam2world --quat_order wxyz
# cam2world/wxyz = the VERIFIED convention (ma_vis body-on-person); world2cam scrambles world poses

In [ ]:
# custom preset: our frame range (0..N-1) + out_dir on Drive
import re
q = open('/content/mamma/configs/examples/presets/quick.yaml').read()
q = re.sub(r'start_frame:\s*\d+', 'start_frame: 0', q)
q = re.sub(r'end_frame:\s*\d+', f'end_frame: {N_SELECT-1}', q)
q = q.replace('out_dir: output', f'out_dir: {MOUT}')
q = q.replace('jobs_log_dir: output/logs/jobs', f'jobs_log_dir: {MOUT}/logs/jobs')
open('/content/mamma/configs/examples/presets/behave.yaml','w').write(q)
print('behave.yaml ready (frames 0..%d, out_dir on Drive)' % (N_SELECT-1))

## 6. Run MAMMA inference (outputs → Drive)

In [ ]:
%cd /content/mamma
CALIB=f'{MAMMA_IN}/{SEQ}.calib.yaml'
!$MM run -r $ROOT -n mamma python -m inference run \
  --cfg configs/examples/presets/behave.yaml \
  --footage "$MAMMA_IN" --seq_name "$SEQ" --calib "$CALIB" --out-tag behave -v
# check ma_vis overlays under $MOUT before trusting ma_3d (body-on-person? else adjust §5 flags)

## 7. MAMMA ma_3d SMPL-X → per-frame exports on Drive

In [ ]:
import glob
cands = glob.glob(f'{MOUT}/**/behave*/**', recursive=True)
print('ma_3d output candidates:'); [print(' ', c) for c in cands[:20]]
# !!! ma_3d/behave (no suffix) = the OLD world2cam SCRAMBLED run — exporting from it silently
# poisons everything downstream (burned the _c2w_hands exports + 2 avatar runs on 2026-07-16).
# Use the cam2world-tagged output; verify against the candidates printed above.
MAMMA_OUT = f'{MOUT}/ma_3d/behave_cam2world'
!cd $FORK && PYTHONPATH=$FORK python -m tracking.behave_to_mamma output \
  --mamma_out_dir "$MAMMA_OUT" --exports_root "$EXPORTS"
# verify the export format the avatar expects: vertices + hand poses
import numpy as np
_d = np.load(sorted(glob.glob(f'{EXPORTS}/frame_*/mamma.npz'))[0])
print('exports ->', EXPORTS)
print('keys:', sorted(_d.keys()))
print('vertices:', 'vertices' in _d, '| hands:', 'left_hand_pose' in _d)
print('frame0 transl:', np.asarray(_d['transl']).round(3), ' (good cam2world run ~ [-0.215, 0.704, 2.362])')

## 8. Next — the avatar (`M10 - surfel_avatar v2`) consumes these exports
```python
MAMMA_ROOT = f'{DRIVE}/datasets/mamma_exports/Date03_Sub05_boxtiny_c2w_hands'   # = EXPORTS
```
(`M10 - smpl_person_render` §6 can also point at this dir — the extra hand-pose keys are ignored there.)